# 02 — Map Universe Tickers to Compustat gvkey

**Goal:** Map each ticker in the S&P 500 historical universe (from notebook 01) to
its corresponding Compustat gvkey, accounting for historical ticker changes
(e.g., FB → META, ANTM → ELV) and ticker collisions across global exchanges.

**Input:** `data/sp500_universe.parquet` (6,535 firm-year rows, 751 tickers)

**Output:** `data/sp500_universe_with_gvkey.parquet`
- One row per (ticker, year, gvkey); gvkey assigned per (ticker, **year**)
- 100% coverage (6,535 rows, all with assigned gvkey)
- 717 unique gvkeys (some tickers map to the same gvkey via historical renames;
  eight lineage tickers legitimately span two gvkeys after corporate events)

**Identifier-mapping approach:**
1. Query `comp.sec_idhist` for historical ticker assignments per gvkey
2. Join to universe with date-overlap matching (each year-end snapshot date must fall
   within the ticker's effective spell), keeping the match **per (ticker, year)**
3. Backfill unmatched years within ticker (e.g., META rows before the META spell
   begins inherit Meta Platforms' gvkey, per the universe's
   forward-corrected-ticker convention)
4. Manually resolve eight edge cases: four tickers with no overlapping spell at all
   (AABA, BTUUQ, VIAV, WYND) and four zombie-spell collisions where an unrelated
   firm's stale spell would win the match (LB, APTV, ES, JEF)

**Revision 2026-06-10:** an earlier version collapsed each ticker to a single gvkey
(`groupby("ticker").first()`), which (a) assigned LB and APTV to unrelated micro-caps
(La Barge Inc; Advanced Promotion Technologies) and ES/JEF to EnergySolutions Inc /
Jefferies Group LLC, silently dropping L Brands, Aptiv, Eversource and
Jefferies Financial from all downstream panels, and (b) truncated lineages that span
two gvkeys after corporate events (AGN, CB, DD, DOW, FOX, FOXA, FTI, IR — dropping
Allergan plc, Chubb Ltd, DuPont/Dow Inc, Fox Corp, TechnipFMC and Ingersoll Rand Inc).
The per-year assignment plus expanded manual overrides fixes both failure modes; a
validation cell asserts that exactly these eight lineage tickers span two gvkeys.

**Mapping quality:**
- No (ticker, year) pair maps to multiple gvkeys (asserted)
- Tax-domiciled US-listed firms correctly identified (Accenture/IRL, Aon/IRL,
  Arch Capital/BMU, Amcor/CHE)
- Historical ticker changes recovered (FB → META, ANTM → ELV, CBS → PARA, etc.)

**Why this approach was needed:** Compustat's current `comp.security` table only
returns each firm's *current* ticker, which misses firms whose tickers changed
during the sample window (FB → META, CBS → VIAC, etc.). The `comp.sec_idhist`
table tracks ticker history per gvkey, allowing recovery of these cases.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import wrds

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"

# Load the universe panel from notebook 01
universe = pd.read_parquet(DATA_PROCESSED / "sp500_universe.parquet")
print(f"Universe: {len(universe):,} firm-year rows, {universe['ticker'].nunique()} unique tickers")

# Get the unique tickers — that's what we'll use to find rcid mappings
universe_tickers = sorted(universe["ticker"].unique())
print(f"First 10 tickers: {universe_tickers[:10]}")
print(f"Last 10 tickers: {universe_tickers[-10:]}")

Universe: 6,535 firm-year rows, 751 unique tickers
First 10 tickers: ['A', 'AABA', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABC', 'ABMD', 'ABNB', 'ABT']
Last 10 tickers: ['XLNX', 'XOM', 'XRAY', 'XRX', 'XYL', 'YUM', 'ZBH', 'ZBRA', 'ZION', 'ZTS']


In [2]:
# Read the WRDS username from ~/.pgpass (4th field) so the notebook runs headlessly
WRDS_USERNAME = next(
    line.split(":")[3]
    for line in (Path.home() / ".pgpass").read_text().splitlines()
    if "wrds" in line
)
db = wrds.Connection(wrds_username=WRDS_USERNAME)

Loading library list...


Done


In [3]:
# Build a SQL-safe list of tickers — wrap each in quotes and join with commas
ticker_list_sql = ",".join([f"'{t}'" for t in universe_tickers])

mapping_query = f"""
    SELECT DISTINCT rcid, ticker, gvkey, cusip, company
    FROM   revelio.company_mapping
    WHERE  ticker IN ({ticker_list_sql})
      AND  rcid IS NOT NULL
"""

mapping = db.raw_sql(mapping_query)

print(f"Universe tickers: {len(universe_tickers)}")
print(f"Tickers matched in Revelio: {mapping['ticker'].nunique()}")
print(f"Total mapping rows (some tickers have multiple rcids): {len(mapping)}")
print(f"Tickers with non-null gvkey: {mapping['gvkey'].notna().sum()}")
print()
mapping.head(10)

Universe tickers: 751
Tickers matched in Revelio: 649
Total mapping rows (some tickers have multiple rcids): 1187
Tickers with non-null gvkey: 959



,rcid,ticker,gvkey,cusip,company
0,420392.0,RSG,112168,760759100,"Republic Services, Inc."
1,254822.0,R,009299,783549108,"Ryder System, Inc."
2,538792.0,DGX,064166,74834L100,"Quest Diagnostics, Inc."
3,313561.0,PHM,226841,E8075H159,Pharma Mar SA
4,1302930.0,WHR,011465,963320106,Whirlpool Corp.
5,96818795.0,PEG,<NA>,Y67983109,Petec Trading & Investment Corp.
6,1257382.0,CTL,352724,G2282G151,CleanTech Lithium Plc
7,5847190.0,RCL,104146,Y72509139,Regional Container Lines Public Co. Ltd.
8,4953341.0,MET,<NA>,<NA>,Meta Estate Trust SA
9,97116006.0,ABT,290751,Y0872X105,BenTre Aquaproduct Import & Export JSC


In [4]:
# Same query but restricted to US firms
# US firms in Compustat have numeric gvkeys; non-US have alphanumeric or different patterns
# US CUSIPs are 9 chars with a leading digit (not letter)

mapping_us_query = f"""
    SELECT DISTINCT rcid, ticker, gvkey, cusip, company, hq_country
    FROM   revelio.company_mapping
    WHERE  ticker IN ({ticker_list_sql})
      AND  rcid IS NOT NULL
      AND  gvkey IS NOT NULL
      AND  hq_country = 'United States'
"""

mapping_us = db.raw_sql(mapping_us_query)

print(f"Universe tickers: {len(universe_tickers)}")
print(f"Tickers matched in Revelio (US only): {mapping_us['ticker'].nunique()}")
print(f"Total mapping rows: {len(mapping_us)}")
print()

# Check the same rows that were suspicious before
print("Checking previously-suspicious tickers (DG, PHM):")
print(mapping_us[mapping_us["ticker"].isin(["DG", "PHM"])])
print()

# Look at the head again to spot-check
print("First 10 rows:")
print(mapping_us.head(10))

Universe tickers: 751
Tickers matched in Revelio (US only): 524
Total mapping rows: 524

Checking previously-suspicious tickers (DG, PHM):
          rcid ticker   gvkey      cusip               company     hq_country
52    243649.0    PHM  008823  745867101      PulteGroup, Inc.  United States
417  1353141.0     DG  004016  256677105  Dollar General Corp.  United States

First 10 rows:
      rcid ticker   gvkey      cusip                                       company     hq_country
0    218.0   CBRE  260774  12504L109                              CBRE Group, Inc.  United States
1   7375.0    MAR  028930  571903202                  Marriott International, Inc.  United States
2  25373.0     SO  009850  842587107                              The Southern Co.  United States
3  29206.0    ETR  007366  29364G103                                 Entergy Corp.  United States
4  34099.0   PCAR  008253  693718108                                  PACCAR, Inc.  United States
5  47589.0   SWKS  0013

In [5]:
# Compare which tickers are in universe but NOT in the US-filtered mapping
unmatched = set(universe_tickers) - set(mapping_us["ticker"].unique())
print(f"Tickers in universe but not in US-only mapping: {len(unmatched)}")
print()

# For these unmatched tickers, check what Revelio shows without the US filter
# but still requiring a non-null gvkey (so we know it's a real Compustat firm)
unmatched_list_sql = ",".join([f"'{t}'" for t in unmatched])

unmatched_check = db.raw_sql(f"""
    SELECT DISTINCT ticker, gvkey, company, hq_country, cusip
    FROM   revelio.company_mapping
    WHERE  ticker IN ({unmatched_list_sql})
      AND  rcid IS NOT NULL
      AND  gvkey IS NOT NULL
    ORDER  BY ticker
""")

print(f"Of {len(unmatched)} unmatched tickers, {unmatched_check['ticker'].nunique()} have non-US Revelio entries with gvkey")
print()
print("Distribution of hq_country for these tickers:")
print(unmatched_check["hq_country"].value_counts().head(20))
print()
print("Sample of unmatched tickers:")
print(unmatched_check.head(20))

Tickers in universe but not in US-only mapping: 227

Of 227 unmatched tickers, 105 have non-US Revelio entries with gvkey

Distribution of hq_country for these tickers:
hq_country
Australia         28
United Kingdom    17
Canada            14
Poland            12
Thailand           8
Philippines        8
Ireland            8
France             6
Switzerland        5
Bermuda            4
Germany            4
Norway             3
Vietnam            3
Netherlands        3
Cyprus             3
Israel             2
South Africa       2
Morocco            2
Italy              2
Indonesia          2
Name: count, dtype: Int64

Sample of unmatched tickers:
   ticker   gvkey                          company   hq_country      cusip
0     ABC  258660                           abc SA        Chile  P3714Y190
1     ABC  313808  ABC Company SpA Società Benefit        Italy  T0R29H117
2     ABC  341194              ABC Motors Co. Ltd.    Mauritius  V00099107
3    ACGL  061302          Arch Capital Grou

In [6]:
# Expand the country filter to include common tax-domicile countries
expanded_countries = [
    "United States", "Ireland", "Bermuda", "United Kingdom",
    "Switzerland", "Netherlands", "Luxembourg", "Cayman Islands",
    "Jersey", "Guernsey"
]

country_list_sql = ",".join([f"'{c}'" for c in expanded_countries])

mapping_expanded_query = f"""
    SELECT DISTINCT rcid, ticker, gvkey, cusip, company, hq_country, exchange_name
    FROM   revelio.company_mapping
    WHERE  ticker IN ({ticker_list_sql})
      AND  rcid IS NOT NULL
      AND  gvkey IS NOT NULL
      AND  hq_country IN ({country_list_sql})
"""

mapping_expanded = db.raw_sql(mapping_expanded_query)

print(f"Tickers matched with expanded filter: {mapping_expanded['ticker'].nunique()}")
print(f"Total rows: {len(mapping_expanded)}")
print()

# How many of these tickers still have duplicates after the expansion?
ticker_counts = mapping_expanded["ticker"].value_counts()
duplicated = ticker_counts[ticker_counts > 1]
print(f"Tickers with multiple matches: {len(duplicated)}")
print()
print("Top 20 tickers with multiple matches:")
print(duplicated.head(20))
print()

# Show the duplicates so we can see if collisions slipped back in
if len(duplicated) > 0:
    print("Sample duplicate rows:")
    print(mapping_expanded[mapping_expanded["ticker"].isin(duplicated.head(10).index)]
          .sort_values("ticker")[["ticker", "company", "hq_country", "exchange_name", "cusip"]])

Tickers matched with expanded filter: 561
Total rows: 591

Tickers with multiple matches: 28

Top 20 tickers with multiple matches:
ticker
AMG     3
DIS     3
COST    2
SRE     2
MET     2
CNC     2
BBY     2
AAL     2
ALK     2
BA      2
ADM     2
HAS     2
ORCL    2
TMO     2
CCL     2
COR     2
EOG     2
AEP     2
BMY     2
AMP     2
Name: count, dtype: Int64

Sample duplicate rows:
    ticker                            company      hq_country            exchange_name      cusip
402    AAL                 Anglo American Plc  United Kingdom    London Stock Exchange  G03764142
528    AAL      American Airlines Group, Inc.   United States                   NASDAQ  02376R102
535    ALK             Alaska Air Group, Inc.   United States  New York Stock Exchange  011659109
504    ALK     Alkemy Capital Investments Plc  United Kingdom    London Stock Exchange  G0174Z105
506    AMG             Atlas Metals Group Plc  United Kingdom    London Stock Exchange  G60524108
364    AMG    Affiliate

In [7]:
us_exchanges = [
    "New York Stock Exchange",
    "NASDAQ",
    "NYSE Arca",
    "NYSE American",
    "Cboe BZX",
    "Cboe BYX",
]

exchange_list_sql = ",".join([f"'{e}'" for e in us_exchanges])

mapping_final_query = f"""
    SELECT DISTINCT rcid, ticker, gvkey, cusip, company, hq_country, exchange_name
    FROM   revelio.company_mapping
    WHERE  ticker IN ({ticker_list_sql})
      AND  rcid IS NOT NULL
      AND  gvkey IS NOT NULL
      AND  exchange_name IN ({exchange_list_sql})
"""

mapping_final = db.raw_sql(mapping_final_query)

print(f"Tickers matched (US exchanges only): {mapping_final['ticker'].nunique()}")
print(f"Total rows: {len(mapping_final)}")
print()

# Verify no duplicates remain
ticker_counts = mapping_final["ticker"].value_counts()
duplicated = ticker_counts[ticker_counts > 1]
print(f"Tickers with multiple matches: {len(duplicated)}")
if len(duplicated) > 0:
    print("\nRemaining duplicates:")
    print(mapping_final[mapping_final["ticker"].isin(duplicated.index)]
          [["ticker", "company", "hq_country", "exchange_name"]]
          .sort_values("ticker"))
print()

# Sanity check: the cases we were worried about earlier
print("Re-verify previously-checked tickers:")
print(mapping_final[mapping_final["ticker"].isin(["DG", "PHM", "ACN", "AON", "ACGL", "AMCR"])]
      [["ticker", "company", "hq_country", "exchange_name", "gvkey"]])
print()

# How many universe firms are now unmatched?
unmatched_final = set(universe_tickers) - set(mapping_final["ticker"].unique())
print(f"Universe firms still unmatched: {len(unmatched_final)}")
print(f"Coverage rate: {len(mapping_final['ticker'].unique()) / len(universe_tickers):.1%}")

Tickers matched (US exchanges only): 545
Total rows: 545

Tickers with multiple matches: 0

Re-verify previously-checked tickers:
    ticker                  company     hq_country            exchange_name   gvkey
51     PHM         PulteGroup, Inc.  United States  New York Stock Exchange  008823
165   ACGL  Arch Capital Group Ltd.        Bermuda                   NASDAQ  061302
208   AMCR                Amcor Plc    Switzerland  New York Stock Exchange  100243
398    ACN            Accenture Plc        Ireland  New York Stock Exchange  143357
427     DG     Dollar General Corp.  United States  New York Stock Exchange  004016
526    AON                  Aon Plc        Ireland  New York Stock Exchange  003221



Universe firms still unmatched: 206
Coverage rate: 72.6%


In [8]:
# What's in Revelio for the 204 unmatched tickers if we ignore the exchange filter?
unmatched_final_list_sql = ",".join([f"'{t}'" for t in unmatched_final])

diagnostic = db.raw_sql(f"""
    SELECT DISTINCT ticker, gvkey, company, hq_country, exchange_name
    FROM   revelio.company_mapping
    WHERE  ticker IN ({unmatched_final_list_sql})
      AND  rcid IS NOT NULL
      AND  gvkey IS NOT NULL
    ORDER  BY ticker
""")

print(f"Of {len(unmatched_final)} unmatched tickers, {diagnostic['ticker'].nunique()} appear in Revelio (any country, any exchange)")
print()
print("Exchange distribution for these matches:")
print(diagnostic["exchange_name"].value_counts().head(20))
print()
print("Sample of US-domiciled but missed firms:")
us_missed = diagnostic[diagnostic["hq_country"] == "United States"]
print(f"  US-domiciled tickers we missed: {us_missed['ticker'].nunique()}")
print(us_missed.head(15))
print()
print("Tickers in universe with NO Revelio entry at all (any country, any exchange):")
never_found = set(unmatched_final) - set(diagnostic["ticker"].unique())
print(f"  Count: {len(never_found)}")
print(f"  Sample: {sorted(never_found)[:30]}")

Of 206 unmatched tickers, 84 appear in Revelio (any country, any exchange)

Exchange distribution for these matches:
exchange_name
ASX                                 24
London Stock Exchange               13
Warsaw Stock Exchange               12
Toronto Stock Exchange               9
Stock Exchange of Thailand           8
Philippine Stock Exchange            8
TSX Venture Exchange                 5
Oslo Exchange                        5
Euronext Paris                       5
XETRA                                4
Johannesburg Securities Exchange     2
Euronext Milan                       2
Casablanca Stock Exchange            2
Ho Chi Minh Stock Exchange           2
Indonesia Exchange                   2
Cyprus Stock Exchange                2
Cboe BZX US Equities Exchange        1
Mauritius Stock Exchange             1
Euronext Amsterdam                   1
Euronext Lisbon                      1
Name: count, dtype: Int64

Sample of US-domiciled but missed firms:
  US-domiciled ticker

In [9]:
# Final filter: US exchanges plus US OTC for delisted firms
us_exchanges_final = [
    "New York Stock Exchange",
    "NASDAQ",
    "NYSE Arca",
    "NYSE American",
    "Cboe BZX",
    "Cboe BYX",
    "Cboe BZX US Equities Exchange",
    "US OTC",
]

exchange_list_sql = ",".join([f"'{e}'" for e in us_exchanges_final])

mapping_final = db.raw_sql(f"""
    SELECT DISTINCT rcid, ticker, gvkey, cusip, company, hq_country, exchange_name
    FROM   revelio.company_mapping
    WHERE  ticker IN ({ticker_list_sql})
      AND  rcid IS NOT NULL
      AND  gvkey IS NOT NULL
      AND  exchange_name IN ({exchange_list_sql})
""")

print(f"Final mapping: {mapping_final['ticker'].nunique()} tickers, {len(mapping_final)} rows")
print(f"Coverage: {mapping_final['ticker'].nunique() / len(universe_tickers):.1%}")
print(f"Duplicates: {(mapping_final['ticker'].value_counts() > 1).sum()}")
print()

# Sanity: confirm CBOE and SBNY are now in
print("Recovered firms:")
print(mapping_final[mapping_final["ticker"].isin(["CBOE", "SBNY"])]
      [["ticker", "company", "exchange_name", "gvkey"]])
print()

# Save it for use in subsequent cells
print(f"Final mapping ready: {len(mapping_final)} firms")

Final mapping: 547 tickers, 547 rows
Coverage: 72.8%
Duplicates: 0

Recovered firms:
    ticker                              company                  exchange_name   gvkey
384   SBNY  Signature Bank (New York, New York)                         US OTC  160776
483   CBOE            Cboe Global Markets, Inc.  Cboe BZX US Equities Exchange  184500

Final mapping ready: 547 firms


In [10]:
ticker_list_sql = ",".join([f"'{t}'" for t in universe_tickers])

comp_mapping = db.raw_sql(f"""
    SELECT DISTINCT
        s.gvkey,
        s.tic AS ticker,
        s.cusip,
        c.conm AS company,
        c.fic AS country_of_incorporation,
        s.exchg AS exchange_code
    FROM   comp.security s
    JOIN   comp.company  c USING (gvkey)
    WHERE  s.tic IN ({ticker_list_sql})
      AND  s.tpci = '0'              -- common stock only
""")

print(f"Tickers matched via Compustat: {comp_mapping['ticker'].nunique()}")
print(f"Total rows: {len(comp_mapping)}")
print()

# Check duplicates
ticker_counts = comp_mapping["ticker"].value_counts()
duplicated = ticker_counts[ticker_counts > 1]
print(f"Tickers with multiple gvkey matches: {len(duplicated)}")
print()

# Distribution of country_of_incorporation
print("Country distribution:")
print(comp_mapping["country_of_incorporation"].value_counts().head(15))
print()

# Sample to spot-check
print("Sample:")
print(comp_mapping.head(20))

Tickers matched via Compustat: 687
Total rows: 687

Tickers with multiple gvkey matches: 0

Country distribution:
country_of_incorporation
USA    649
IRL     16
BMU      7
GBR      4
CHE      4
JEY      2
NLD      2
CUW      1
LBR      1
VGB      1
Name: count, dtype: Int64

Sample:
     gvkey ticker      cusip                       company country_of_incorporation  exchange_code
0   001045    AAL  02376R102   AMERICAN AIRLINES GROUP INC                      USA             14
1   001075    PNW  723484101    PINNACLE WEST CAPITAL CORP                      USA             11
2   001078    ABT  002824100           ABBOTT LABORATORIES                      USA             11
3   001161    AMD  007903107        ADVANCED MICRO DEVICES                      USA             14
4   001177    AET  00817Y108                     AETNA INC                      USA             11
5   001209    APD  009158106  AIR PRODUCTS & CHEMICALS INC                      USA             11
6   001230    ALK  0116

In [11]:
unmatched_compustat = set(universe_tickers) - set(comp_mapping["ticker"].unique())
print(f"Still unmatched: {len(unmatched_compustat)}")
print(f"Sample: {sorted(unmatched_compustat)[:30]}")
print()

# What does Compustat have for these tickers if we relax the common-stock filter?
unmatched_sql = ",".join([f"'{t}'" for t in unmatched_compustat])

relax_check = db.raw_sql(f"""
    SELECT DISTINCT s.gvkey, s.tic AS ticker, s.tpci, c.conm AS company
    FROM   comp.security s
    JOIN   comp.company  c USING (gvkey)
    WHERE  s.tic IN ({unmatched_sql})
      AND  c.fic IN ('USA','IRL','BMU','GBR','CHE','JEY','NLD','CYM')
    ORDER BY ticker
    LIMIT 30
""")

print("Sample of relaxed-filter results:")
print(relax_check)
print()
print("tpci distribution:")
print(relax_check["tpci"].value_counts())

Still unmatched: 64
Sample: ['ABC', 'ADS', 'ANTM', 'BHGE', 'BIG', 'BK', 'BLL', 'CA', 'CAM', 'CBS', 'CCE', 'CDAY', 'CHK', 'COG', 'CTL', 'DF', 'DISCA', 'DNR', 'DTV', 'DWDP', 'EMC', 'ENDP', 'ESV', 'FB', 'FBHS', 'FI', 'FII', 'FLT', 'FRC', 'FTR']



Sample of relaxed-filter results:
    gvkey ticker tpci                       company
0  045004     CA    %  XTRACKERS CALIFORNIA MUNI BD
1  074694    CAM    %  AB CALIFORNIA INTERMDT MNCPL
2  003897    DTV    4                 DTE ENERGY CO
3  042939    EMC    %  EMERGING MARKET CONSUMER ETF
4  051995     FB    %  PROSHARES S&P 500 DYN BF ETF
5  050783   INFO    %  HARBOR PNAGR DYNA L CP C ETF
6  061110    PCL    %  PGIM CORPORATE BOD 10 YR ETF
7  051443    POM    F            POMDOCTOR LTD -ADR
8  032546     SE    F                   SEA LIMITED
9  075344   SPLS    %  PIMCO US STOCKS PLUS ACTI BD

tpci distribution:
tpci
%    7
F    2
4    1
Name: count, dtype: Int64


In [12]:
ticker_list_sql = ",".join([f"'{t}'" for t in universe_tickers])

# comp.names has historical ticker/name associations
hist_mapping = db.raw_sql(f"""
    SELECT DISTINCT
        n.gvkey,
        n.tic AS ticker,
        n.conm AS company_historical,
        c.conm AS company_current,
        c.fic AS country_of_incorporation
    FROM   comp.names n
    JOIN   comp.company c USING (gvkey)
    WHERE  n.tic IN ({ticker_list_sql})
""")

print(f"Tickers matched via comp.names: {hist_mapping['ticker'].nunique()}")
print(f"Total rows: {len(hist_mapping)}")
print()

# Did we recover the historically-renamed firms?
recovered = ["FB", "EMC", "CBS", "ANTM", "BHGE", "DTV", "DWDP", "GGP", "FRC"]
print("Recovery check for renamed/merged firms:")
print(hist_mapping[hist_mapping["ticker"].isin(recovered)]
      [["ticker", "gvkey", "company_historical", "company_current"]])
print()

# How many tickers still missing?
still_missing = set(universe_tickers) - set(hist_mapping["ticker"].unique())
print(f"Still unmatched after using comp.names: {len(still_missing)}")
print(f"Sample: {sorted(still_missing)[:30]}")
print()

# Check for new duplicate problems
ticker_counts = hist_mapping["ticker"].value_counts()
duplicated = ticker_counts[ticker_counts > 1]
print(f"Tickers with multiple gvkey matches: {len(duplicated)}")
if len(duplicated) > 0:
    print(duplicated.head(10))

Tickers matched via comp.names: 682
Total rows: 682

Recovery check for renamed/merged firms:
    ticker   gvkey            company_historical               company_current
13      FB  051995  PROSHARES S&P 500 DYN BF ETF  PROSHARES S&P 500 DYN BF ETF
509    EMC  042939  EMERGING MARKET CONSUMER ETF  EMERGING MARKET CONSUMER ETF

Still unmatched after using comp.names: 69
Sample: ['ABC', 'ADS', 'ANTM', 'BHGE', 'BIG', 'BK', 'BLL', 'BTUUQ', 'CBS', 'CCE', 'CDAY', 'CHK', 'COG', 'CTL', 'CVC', 'DF', 'DISCA', 'DISCK', 'DNR', 'DTV', 'DWDP', 'ENDP', 'ESV', 'FBHS', 'FI', 'FII', 'FLT', 'FOX', 'FRC', 'FRX']

Tickers with multiple gvkey matches: 0


In [13]:
# Check if comp.sec_history exists and what's in it
try:
    desc = db.describe_table(library='comp', table='sec_history')
    print("comp.sec_history columns:")
    print(desc[['name', 'type', 'comment']].to_string())
    print()
    sample = db.get_table(library='comp', table='sec_history', obs=5)
    print("Sample rows:")
    print(sample)
except Exception as e:
    print(f"comp.sec_history NOT AVAILABLE: {type(e).__name__}")
    print(f"  Detail: {str(e)[:300]}")

print()
print("---")
print()

# Search comp library for tables with 'hist' or 'history' in the name
try:
    tables = db.list_tables(library='comp')
    relevant = [t for t in tables if any(k in t.lower() for k in ['hist', 'history', 'sec'])]
    print("Tables in comp matching hist/history/sec patterns:")
    for t in sorted(relevant):
        print(f"  comp.{t}")
except Exception as e:
    print(f"Search failed: {e}")

Approximately 325420 rows in comp.sec_history.


comp.sec_history columns:
        name         type                          comment
0      gvkey   VARCHAR(7)               Global Company Key
1        iid   VARCHAR(4)                  Global Issue ID
2       item  VARCHAR(21)       Standardized Item Mnemonic
3  itemvalue  VARCHAR(21)  Security Historical Identifiers
4    effdate         DATE              Effective From Date
5   thrudate         DATE              Effective Thru Date

Sample rows:
    gvkey iid        item itemvalue     effdate thrudate
0  001000  01  PRIHISTUSA        01  1900-01-01     <NA>
1  001000  01   MKVALINCL         Y  1900-01-01     <NA>
2  001001  01  PRIHISTUSA        01  1900-01-01     <NA>
3  001001  01   MKVALINCL         Y  1900-01-01     <NA>
4  001002  01  PRIHISTUSA        01  1900-01-01     <NA>

---



Tables in comp matching hist/history/sec patterns:
  comp.asec_amda
  comp.asec_imda
  comp.asec_notesa
  comp.asec_notesq
  comp.asec_transa
  comp.asec_transq
  comp.co_acthist
  comp.g_sec_adesind
  comp.g_sec_adjfact
  comp.g_sec_afnd
  comp.g_sec_afnddc
  comp.g_sec_afnt
  comp.g_sec_divid
  comp.g_sec_dprc
  comp.g_sec_dtrt
  comp.g_sec_gmdivfn
  comp.g_sec_gmth
  comp.g_sec_gmthdiv
  comp.g_sec_gmthprc
  comp.g_sec_history
  comp.g_sec_idesind
  comp.g_sec_ifnd
  comp.g_sec_ifnt
  comp.g_sec_split
  comp.g_secd
  comp.g_secm
  comp.g_secnamesd
  comp.g_security
  comp.gsecnamesm
  comp.r_indsec
  comp.r_sec_stat
  comp.r_secannfn
  comp.r_sectors
  comp.sec_adesind
  comp.sec_adjfact
  comp.sec_afnd
  comp.sec_afnddc
  comp.sec_afnt
  comp.sec_divid
  comp.sec_dprc
  comp.sec_dtrt
  comp.sec_history
  comp.sec_idcurrent
  comp.sec_idesind
  comp.sec_idhist
  comp.sec_ifnd
  comp.sec_ifnt
  comp.sec_mdivfn
  comp.sec_mshare
  comp.sec_msptfn
  comp.sec_mth
  comp.sec_mthdiv
  com

In [14]:
try:
    desc = db.describe_table(library='comp', table='sec_idhist')
    print("comp.sec_idhist columns:")
    print(desc[['name', 'type', 'comment']].to_string())
    print()
    sample = db.get_table(library='comp', table='sec_idhist', obs=10)
    print("Sample rows:")
    print(sample)
except Exception as e:
    print(f"NOT AVAILABLE: {e}")

Approximately 850684 rows in comp.sec_idhist.


comp.sec_idhist columns:
        name         type                                   comment
0      gvkey   VARCHAR(7)  Global Company Key - Security ID History
1        iid   VARCHAR(4)            Issue Id - Security ID History
2       item  VARCHAR(11)                  Security Identifier Type
3  itemvalue  VARCHAR(21)                 Security Identifier Value
4    efffrom         DATE           Effective From Date of Security
5    effthru         DATE           Effective Thru Date of Security



Sample rows:
    gvkey iid   item     itemvalue     efffrom     effthru
0  001000  01    TIC           AE.  1987-04-30  1991-01-23
1  001000  01    TIC          AE.2  1998-05-01  2900-01-01
2  001000  01  CUSIP     000032102  1961-12-31  2900-01-01
3  001001  01    TIC          AMFD  1987-04-30  1990-08-30
4  001001  01    TIC         AMFD.  1990-08-31  2900-01-01
5  001001  01  CUSIP     000165100  1983-09-20  2900-01-01
6  001001  01   ISIN  US0001651001  2013-01-01  2900-01-01
7  001002  01    TIC          AAIC  1988-09-08  2020-10-25
8  001002  01    TIC        AAIC.1  2020-10-26  2900-01-01
9  001002  01  CUSIP     000352104  1960-12-31  2900-01-01


In [15]:
ticker_list_sql = ",".join([f"'{t}'" for t in universe_tickers])

hist_query = f"""
    SELECT DISTINCT
        h.gvkey,
        h.itemvalue AS ticker,
        h.efffrom,
        h.effthru,
        c.conm AS company_current,
        c.fic AS country
    FROM   comp.sec_idhist h
    JOIN   comp.company    c USING (gvkey)
    WHERE  h.item = 'TIC'
      AND  h.itemvalue IN ({ticker_list_sql})
      AND  h.efffrom <= '2024-12-31'
      AND  (h.effthru >= '2012-01-01' OR h.effthru = '2900-01-01')
"""

hist_mapping = db.raw_sql(hist_query)

print(f"Tickers matched via sec_idhist: {hist_mapping['ticker'].nunique()}")
print(f"Total rows: {len(hist_mapping)}")
print(f"Unique gvkeys: {hist_mapping['gvkey'].nunique()}")
print()

# Did we recover the renamed firms?
recovered_check = ["FB", "EMC", "CBS", "ANTM", "BHGE", "DTV", "DWDP", "FRC",
                   "CELG", "CERN", "AET", "ATVI", "CTXS"]
print("Recovery check for previously-missing renamed/merged firms:")
print(hist_mapping[hist_mapping["ticker"].isin(recovered_check)]
      [["ticker", "gvkey", "company_current", "efffrom", "effthru"]]
      .sort_values("ticker"))
print()

# Tickers still missing
still_missing = set(universe_tickers) - set(hist_mapping["ticker"].unique())
print(f"Still unmatched: {len(still_missing)}")
print(f"Sample: {sorted(still_missing)[:30]}")
print()

# Check for duplicates (multiple gvkeys per ticker — would indicate collision)
ticker_gvkey_pairs = hist_mapping.groupby("ticker")["gvkey"].nunique()
ambig = ticker_gvkey_pairs[ticker_gvkey_pairs > 1]
print(f"Tickers with multiple gvkeys (potential collisions): {len(ambig)}")
if len(ambig) > 0:
    print("Sample:")
    print(hist_mapping[hist_mapping["ticker"].isin(ambig.head(10).index)]
          [["ticker", "gvkey", "company_current", "efffrom", "effthru"]]
          .sort_values(["ticker", "efffrom"]))

Tickers matched via sec_idhist: 751
Total rows: 850
Unique gvkeys: 787

Recovery check for previously-missing renamed/merged firms:
    ticker   gvkey               company_current     efffrom     effthru
4      AET  001177                     AETNA INC  1986-12-14  2900-01-01
722   ANTM  145046           ELEVANCE HEALTH INC  2014-12-03  2022-06-27
816   ATVI  180405       ACTIVISION BLIZZARD INC  2008-08-07  2900-01-01
560   BHGE  032106               BAKER HUGHES CO  2017-07-05  2019-10-17
375    CBS  013714       PARAMOUNT SKYDANCE CORP  2006-01-03  2019-12-04
371   CELG  013599                  CELGENE CORP  1987-11-30  2900-01-01
354   CERN  012850                   CERNER CORP  1987-03-31  2900-01-01
616   CTXS  061676            CITRIX SYSTEMS INC  1996-03-14  2900-01-01
82     DTV  003897                 DTE ENERGY CO  2016-10-11  2900-01-01
342    DTV  012206                       DIRECTV  2004-03-17  2016-10-06
89    DWDP  004060         DUPONT DE NEMOURS INC  2017-09-01  201

In [16]:
# Replace the 2900-01-01 sentinel with NaT (which represents "still active" / open-ended)
hist_mapping["efffrom"] = pd.to_datetime(hist_mapping["efffrom"])
hist_mapping["effthru"] = pd.to_datetime(
    hist_mapping["effthru"].astype(str).replace("2900-01-01", pd.NA),
    errors="coerce"
)

print(f"Effthru NaT count (still-active spells): {hist_mapping['effthru'].isna().sum()}")
print(f"Effthru range (excluding NaT): {hist_mapping['effthru'].min()} to {hist_mapping['effthru'].max()}")
print()

# Now the date-overlap join
universe["snapshot_date"] = pd.to_datetime(universe["snapshot_date"])
merged = universe.merge(hist_mapping, on="ticker", how="left")

# Keep rows where the ticker spell covers the universe year-end snapshot
# NaT in effthru means "still active" — should be treated as never expired
mask = (
    (merged["snapshot_date"] >= merged["efffrom"]) &
    (merged["effthru"].isna() | (merged["snapshot_date"] <= merged["effthru"]))
)
matched = merged[mask].copy()

print(f"Universe rows: {len(universe):,}")
print(f"Matched rows: {len(matched):,}")
print(f"Unmatched universe rows: {len(universe) - len(matched):,}")
print()

# Same diagnostics as before
universe_pairs = set(zip(universe["ticker"], universe["year"]))
matched_pairs = set(zip(matched["ticker"], matched["year"]))
unmatched_pairs = universe_pairs - matched_pairs
print(f"Universe (ticker, year) pairs: {len(universe_pairs):,}")
print(f"Matched (ticker, year) pairs: {len(matched_pairs):,}")
print(f"Unmatched (ticker, year) pairs: {len(unmatched_pairs)}")
if unmatched_pairs:
    print(f"Sample unmatched: {sorted(unmatched_pairs)[:20]}")

print()
dup_check = matched.groupby(["ticker", "year"])["gvkey"].nunique()
problem_pairs = dup_check[dup_check > 1]
print(f"(ticker, year) pairs with multiple gvkey matches: {len(problem_pairs)}")
if len(problem_pairs) > 0:
    print("Sample of duplicates:")
    sample = matched[matched.set_index(["ticker", "year"]).index.isin(problem_pairs.head(10).index)]
    print(sample[["ticker", "year", "gvkey", "company_current", "efffrom", "effthru"]]
          .sort_values(["ticker", "year"]))

Effthru NaT count (still-active spells): 685
Effthru range (excluding NaT): 2012-02-15 00:00:00 to 2026-05-20 00:00:00

Universe rows: 6,535
Matched rows: 6,444
Unmatched universe rows: 91

Universe (ticker, year) pairs: 6,535
Matched (ticker, year) pairs: 6,444
Unmatched (ticker, year) pairs: 91
Sample unmatched: [('AABA', 2012), ('AABA', 2013), ('AABA', 2014), ('AABA', 2015), ('AABA', 2016), ('ANDV', 2012), ('ANDV', 2013), ('ANDV', 2014), ('ANDV', 2015), ('ANDV', 2016), ('ANTM', 2012), ('ANTM', 2013), ('ARNC', 2012), ('ARNC', 2013), ('ARNC', 2014), ('ARNC', 2015), ('BHGE', 2012), ('BHGE', 2013), ('BHGE', 2014), ('BHGE', 2015)]

(ticker, year) pairs with multiple gvkey matches: 0


In [17]:
# Assign gvkey per (ticker, YEAR) from the spell-overlap match.
#
# FIX 2026-06-10: an earlier version collapsed to one gvkey per ticker with
# groupby("ticker").first(). Under the universe's forward-corrected-ticker
# convention that silently picked stale spells from unrelated firms
# (LB -> La Barge instead of L Brands; APTV -> Advanced Promotion Technologies
# instead of Aptiv/Delphi) and truncated lineages that legitimately span two
# gvkeys after corporate events (CB: Chubb Corp -> Chubb Ltd; DD/DOW: DowDuPont
# split; FOX/FOXA: 21st Century Fox -> Fox Corp; FTI: FMC Technologies ->
# TechnipFMC; IR: Trane lineage -> Ingersoll Rand Inc; AGN: Allergan Inc ->
# Allergan plc). The per-year spell match is correct for these — keep it.

# The dup check above guarantees one gvkey per (ticker, year)
assert matched.groupby(["ticker", "year"])["gvkey"].nunique().max() == 1
year_gvkey = (
    matched.groupby(["ticker", "year"])["gvkey"]
    .first()
    .rename("gvkey_matched")
    .reset_index()
)

universe_with_gvkey = universe.merge(year_gvkey, on=["ticker", "year"], how="left")

# Backfill within ticker: years with no overlapping spell (forward-corrected
# renames, e.g. META rows before the META spell begins) inherit the gvkey from
# the nearest LATER matched year; trailing gaps inherit the previous one.
universe_with_gvkey = universe_with_gvkey.sort_values(["ticker", "year"])
universe_with_gvkey["gvkey"] = (
    universe_with_gvkey.groupby("ticker")["gvkey_matched"]
    .transform(lambda s: s.bfill().ffill())
)
universe_with_gvkey = universe_with_gvkey.drop(columns="gvkey_matched").reset_index(drop=True)

print(f"Universe rows: {len(universe_with_gvkey):,}")
print(f"Rows with gvkey assigned: {universe_with_gvkey['gvkey'].notna().sum():,}")
print(f"Rows still missing gvkey: {universe_with_gvkey['gvkey'].isna().sum()}")
print()

# Spot-check the renamed-firm recoveries
print("Recovered renamed firms (should have gvkey):")
print(universe_with_gvkey[universe_with_gvkey["ticker"].isin(["ANDV", "ANTM", "ARNC", "BHGE", "META"])]
      .groupby("ticker")[["gvkey"]]
      .first())
print()

print(f"Unique gvkeys in panel: {universe_with_gvkey['gvkey'].nunique()}")

# Tickers without any gvkey (no spell ever overlaps a snapshot) -> manual mapping below
still_missing = universe_with_gvkey[universe_with_gvkey["gvkey"].isna()]["ticker"].unique()
print(f"Tickers without any gvkey assignment: {len(still_missing)}")
if len(still_missing) > 0:
    print(f"  Tickers: {sorted(still_missing)}")

Universe rows: 6,535
Rows with gvkey assigned: 6,521
Rows still missing gvkey: 14

Recovered renamed firms (should have gvkey):
         gvkey
ticker        
ANDV    010466
ANTM    145046
ARNC    028192
BHGE    032106
META    170617

Unique gvkeys in panel: 717
Tickers without any gvkey assignment: 4
  Tickers: ['AABA', 'BTUUQ', 'VIAV', 'WYND']


In [18]:
# These four tickers came up empty — investigate
diag = db.raw_sql("""
    SELECT h.gvkey, h.itemvalue AS ticker, h.efffrom, h.effthru,
           c.conm AS company_current
    FROM   comp.sec_idhist h
    JOIN   comp.company    c USING (gvkey)
    WHERE  h.item = 'TIC'
      AND  h.itemvalue IN ('AABA', 'BTUUQ', 'VIAV', 'WYND')
""")
print("Direct lookup in sec_idhist:")
print(diag)
print()

# Try looking by company name (case-insensitive substring match)
# NB: %% because sqlalchemy's exec_driver_sql treats bare % as a parameter marker
name_search = db.raw_sql("""
    SELECT gvkey, conm, fic, ipodate
    FROM   comp.company
    WHERE  LOWER(conm) LIKE '%%altaba%%'
       OR  LOWER(conm) LIKE '%%viavi%%'
       OR  LOWER(conm) LIKE '%%wyndham%%'
       OR  LOWER(conm) LIKE '%%peabody energy%%'
    ORDER BY conm
""")
print("Direct lookup by company name in comp.company:")
print(name_search)

Direct lookup in sec_idhist:
    gvkey ticker     efffrom     effthru         company_current
0  024830   WYND  2009-06-22  2011-10-16          WYNDSTORM CORP
1  062634   AABA  2017-06-19  2900-01-01              ALTABA INC
2  174729   WYND  2018-06-01  2021-02-16  TRAVEL PLUS LEISURE CO
3  029241   VIAV  2015-08-04  2900-01-01     VIAVI SOLUTIONS INC
4  142460  BTUUQ  2016-04-14  2900-01-01     PEABODY ENERGY CORP



Direct lookup by company name in comp.company:
    gvkey                       conm  fic     ipodate
0  062634                 ALTABA INC  USA  1996-04-12
1  023698                ALTABANCORP  USA  2015-06-11
2  142460        PEABODY ENERGY CORP  USA        <NA>
3  127656    VIAVID BROADCASTING INC  USA  1999-12-27
4  029241        VIAVI SOLUTIONS INC  USA  1993-11-17
5  062909         WYNDHAM HOTEL CORP  USA  1996-05-20
6  033243    WYNDHAM HOTELS & RESRTS  USA        <NA>
7  061352  WYNDHAM INTERNATIONAL INC  USA  1995-09-27


In [19]:
# Manual ticker-level overrides (applied with PRIORITY over the spell match).
#
# Class 1 — no sec_idhist spell overlaps any snapshot date:
#   AABA, BTUUQ, VIAV, WYND (as before).
#
# Class 2 — zombie-spell collisions (FIX 2026-06-10, verified against comp.names,
# crsp.ccmxpf_linktable and fja05680 membership continuity): an unrelated firm's
# stale ticker spell wins the overlap match for early snapshot years:
#   LB   -> 006733 Bath & Body Works (Limited Brands / L Brands lineage).
#           006534 = La Barge Inc, defense micro-cap, never in this universe.
#   APTV -> 118122 Aptiv PLC (Delphi Automotive lineage; DLPH spell 2011-2017).
#           024217 = Advanced Promotion Technologies, delisted 1996.
#   ES   -> 007970 Eversource Energy (Northeast Utilities lineage).
#           178846 = EnergySolutions Inc, never an S&P 500 member.
#   JEF  -> 006682 Jefferies Financial Group (Leucadia National lineage).
#           006239 = Jefferies Group LLC, Leucadia subsidiary after 03/2013.

manual_mapping = {
    "AABA": "062634",    # Altaba Inc
    "BTUUQ": "142460",   # Peabody Energy Corp
    "VIAV": "029241",    # Viavi Solutions Inc
    "WYND": "174729",    # Travel + Leisure Co (the S&P 500 firm; ignore Wyndstorm)
    "LB": "006733",      # L Brands / Bath & Body Works (NOT La Barge)
    "APTV": "118122",    # Aptiv PLC, formerly Delphi Automotive (NOT Advanced Promotion Tech)
    "ES": "007970",      # Eversource Energy, formerly Northeast Utilities (NOT EnergySolutions)
    "JEF": "006682",     # Jefferies Financial Group, formerly Leucadia (NOT Jefferies Group LLC)
}

override = universe_with_gvkey["ticker"].map(manual_mapping)
n_affected = (override.notna() & (override != universe_with_gvkey["gvkey"])).sum()
universe_with_gvkey["gvkey"] = override.fillna(universe_with_gvkey["gvkey"])

print(f"Rows changed or filled by manual mapping: {n_affected}")
print(f"Rows with gvkey: {universe_with_gvkey['gvkey'].notna().sum():,} / {len(universe_with_gvkey):,}")
print(f"Coverage: {universe_with_gvkey['gvkey'].notna().sum() / len(universe_with_gvkey):.1%}")
print(f"Unique gvkeys: {universe_with_gvkey['gvkey'].nunique()}")
print()

print("Manual overrides check:")
print(universe_with_gvkey[universe_with_gvkey["ticker"].isin(manual_mapping.keys())]
      .groupby("ticker")[["gvkey"]].first())
print()

# Validation: full coverage, and the ONLY tickers mapping to >1 gvkey across
# years are the eight known post-corporate-event lineages.
assert universe_with_gvkey["gvkey"].notna().all(), "unmapped universe rows remain"
span = universe_with_gvkey.groupby("ticker")["gvkey"].nunique()
multi = set(span[span > 1].index)
expected_lineages = {"AGN", "CB", "DD", "DOW", "FOX", "FOXA", "FTI", "IR"}
assert multi == expected_lineages, f"unexpected multi-gvkey tickers: {multi ^ expected_lineages}"

print("Lineage tickers spanning two gvkeys (corporate events, intentional):")
print(universe_with_gvkey[universe_with_gvkey["ticker"].isin(sorted(multi))]
      .groupby(["ticker", "gvkey"])["year"].agg(["min", "max"]))

Rows changed or filled by manual mapping: 10
Rows with gvkey: 6,535 / 6,535
Coverage: 100.0%
Unique gvkeys: 717

Manual overrides check:
         gvkey
ticker        
AABA    062634
APTV    118122
BTUUQ   142460
ES      007970
JEF     006682
LB      006733
VIAV    029241
WYND    174729

Lineage tickers spanning two gvkeys (corporate events, intentional):
                min   max
ticker gvkey             
AGN    015708  2012  2014
       027845  2015  2019
CB     003024  2012  2015
       028034  2016  2024
DD     004060  2019  2024
       004087  2012  2016
DOW    004060  2012  2016
       034443  2019  2024
FOX    012886  2015  2018
       034636  2019  2024
FOXA   012886  2012  2018
       034636  2019  2024
FTI    030923  2017  2020
       142811  2012  2016
IR     005959  2012  2019
       030098  2020  2024


In [20]:
# Save the final universe-with-gvkey panel
output_path = DATA_PROCESSED / "sp500_universe_with_gvkey.parquet"
universe_with_gvkey.to_parquet(output_path, index=False)

# Verify by reading back
check = pd.read_parquet(output_path)
print(f"Saved {len(check):,} rows to {output_path}")
print(f"Columns: {list(check.columns)}")
print(f"Unique gvkeys: {check['gvkey'].nunique()}")
print(f"Years: {sorted(check['year'].unique())}")
print()
check.head()

Saved 6,535 rows to /Users/<wrds-username>/thesis/data/sp500_universe_with_gvkey.parquet
Columns: ['year', 'ticker', 'snapshot_date', 'gvkey']
Unique gvkeys: 717
Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]



,year,ticker,snapshot_date,gvkey
0,2012,A,2012-12-28,126554
1,2013,A,2013-12-31,126554
2,2014,A,2014-12-24,126554
3,2015,A,2015-12-31,126554
4,2016,A,2016-12-30,126554
